# Document Ingestion and Preprocessing Pipeline
**Author:** Juan Esteban Agudelo Ortiz  
**Email:** juan.es.agor@gmail.com

---

This notebook implements a multi-branch ingestion pipeline that accepts
three input types: PDF files, images of handwritten notes, and reference
images. The output is a clean text representation suitable for downstream
language model processing.

### Limitations
1. PDF extraction degrades on scanned PDFs; these require OCR treatment.
2. PaddleOCR accuracy is sensitive to image quality; low-resolution photos produce noisy text.
3. Multi-column PDF layouts may produce interleaved text despite block-based extraction.
4. Mathematical equations in PDFs are extracted as text approximations; LaTeX source is not recovered.

## 0. Install dependencies

In [7]:
# Run only once
# !pip install pymupdf paddleocr paddlepaddle pillow numpy

## 1. Imports and configuration

In [8]:
import re
from pathlib import Path
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional

import fitz  # pymupdf
import numpy as np
from PIL import Image
from paddleocr import PaddleOCR

# --- Directory setup ---
BASE_DIR    = Path("..")
UPLOADS_DIR = BASE_DIR / "data" / "uploads"
OUT_DIR     = BASE_DIR / "outputs"

UPLOADS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Uploads dir : {UPLOADS_DIR.resolve()}")
print(f"Outputs dir : {OUT_DIR.resolve()}")
print("Configuration ready ✓")

Uploads dir : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/uploads
Outputs dir : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/outputs
Configuration ready ✓


## 2. Input type classification

The pipeline branches on input type. Classification is based on file extension
and a user-provided flag for images. The three branches are mutually exclusive:
a file goes through exactly one branch.

| Input type | Branch | Output |
|---|---|---|
| `.pdf` | PDF extraction | ordered text string |
| `.jpg`, `.png` (notes) | OCR | text string |
| `.jpg`, `.png` (reference) | passthrough | PIL Image object |
| `.txt` | direct read | text string |

In [9]:
class InputType(Enum):
    """
    Enum for classifying input types.
    """
    PDF           = "pdf"
    HANDWRITTEN   = "handwritten_image"
    REFERENCE_IMG = "reference_image"
    PLAIN_TEXT    = "plain_text"


def classify_input(file_path: Path, is_reference_image: bool = False) -> InputType:
    """
    Classify a file into one of the four input types.

    Parameters
    ----------
    file_path : Path
        Path to the uploaded file.
    is_reference_image : bool
        If True and the file is an image, treat it as a reference image
        (passthrough) rather than handwritten notes (OCR).

    Returns
    -------
    InputType
        The classified input type.
    """
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        return InputType.PDF
    elif suffix == ".txt":
        return InputType.PLAIN_TEXT
    elif suffix in (".jpg", ".jpeg", ".png", ".webp"):
        if is_reference_image:
            return InputType.REFERENCE_IMG
        return InputType.HANDWRITTEN
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


# Quick test
test_cases = [
    (Path("notes.jpg"),    False, InputType.HANDWRITTEN),
    (Path("amide.png"),    True,  InputType.REFERENCE_IMG),
    (Path("chapter3.pdf"), False, InputType.PDF),
    (Path("summary.txt"),  False, InputType.PLAIN_TEXT),
]

for path, is_ref, expected in test_cases:
    result = classify_input(path, is_ref)
    status = "✓" if result == expected else "✗"
    print(f"{status} {path.name:20s} → {result.value}")


✓ notes.jpg            → handwritten_image
✓ amide.png            → reference_image
✓ chapter3.pdf         → pdf
✓ summary.txt          → plain_text


## 3. PDF text extraction

PyMuPDF extracts text block-by-block, where each block corresponds to a
typographic unit on the page: paragraph, heading, or caption. Blocks are
sorted by vertical position $y_0$ then horizontal position $x_0$:

$$\text{sort key} = (y_0, x_0)$$

This preserves reading order for single-column layouts. For two-column
layouts, a column-aware sort would be needed (see Limitations).

In [10]:
def extract_text_from_pdf(pdf_path: Path, min_block_chars: int = 20) -> str:
    """
    Extract ordered text from a PDF using block-based extraction.

    Parameters
    ----------
    pdf_path : Path
        Path to the PDF file.
    min_block_chars : int
        Minimum character count for a block to be included.
        Filters out page numbers, headers, and single-word noise.

    Returns
    -------
    str
        Clean, ordered text extracted from all pages.
    """
    doc = fitz.open(pdf_path)
    all_text = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("blocks")

        # Sort blocks by vertical then horizontal position
        blocks_sorted = sorted(blocks, key=lambda b: (b[1], b[0]))

        page_text = []
        for block in blocks_sorted:
            text = block[4].strip()

            # Skip short blocks: page numbers, headers, noise
            if len(text) < min_block_chars:
                continue

            text = re.sub(r"\s+", " ", text)
            page_text.append(text)

        if page_text:
            all_text.append(f"--- Page {page_num + 1} ---\n" + "\n\n".join(page_text))

    doc.close()
    return "\n\n".join(all_text)

## 4. OCR for handwritten notes

PaddleOCR uses a three-stage pipeline internally:
1. **Detection:** locates text regions in the image
2. **Recognition:** reads the text within each detected region
3. **Direction classification:** determines text orientation

Each recognized word carries a confidence score $c \in [0, 1]$.
Low-confidence recognitions are filtered with a threshold $\tau$:

$$\text{keep word} \iff c \geq \tau$$

A typical value is $\tau = 0.7$. Lower values include more text at
the cost of higher error rate.

In [12]:
# Initialize OCR once — model downloads on first run (~100MB)
# use_angle_cls=True handles rotated text in phone photos
ocr_engine = PaddleOCR(use_angle_cls=True, lang="es")


def extract_text_from_image(image_path: Path, confidence_threshold: float = 0.7) -> str:
    """
    Extract text from a handwritten note image using OCR.

    Parameters
    ----------
    image_path : Path
        Path to the image file.
    confidence_threshold : float
        Minimum confidence score to include a recognized word.
        Range [0, 1]. Default 0.7.

    Returns
    -------
    str
        Extracted text, one line per detected text region.
    """
    result = ocr_engine.ocr(str(image_path), cls=True)

    if result is None or result[0] is None:
        return ""

    lines = []
    for line in result[0]:
        text, confidence = line[1][0], line[1][1]
        if confidence >= confidence_threshold:
            lines.append(text)

    return "\n".join(lines)


print("OCR engine initialized ✓")
print("To test: call extract_text_from_image(Path('your_notes.jpg'))")

/tmp/ipykernel_7726/1333344312.py:3: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr_engine = PaddleOCR(use_angle_cls=True, lang="es")
/home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/.venv/lib/python3.13/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/home/juanessao/.paddlex/offici

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/home/juanessao/.paddlex/official_models/UVDoc`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in `/home/juanessao/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.


...

Creating model: ('PP-OCRv5_server_det', None, None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `/home/juanessao/.paddlex/official_models/PP-OCRv5_server_det`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('latin_PP-OCRv5_mobile_rec', None, None)
Using official model (latin_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in `/home/juanessao/.paddlex/official_models/latin_PP-OCRv5_mobile_rec`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

OCR engine initialized ✓
To test: call extract_text_from_image(Path('your_notes.jpg'))


## 5. Reference image handling

Reference images are not processed for text. They are loaded, resized to a
standard maximum dimension to limit memory usage, and passed through to the
PDF generation stage as PIL Image objects.

The resize operation preserves aspect ratio. For an image with dimensions
$(W, H)$ and maximum allowed dimension $D_{max}$, the scale factor is:

$$s = \min\left(1.0, \frac{D_{max}}{\max(W, H)}\right)$$

Images smaller than $D_{max}$ are left unchanged ($s = 1.0$).
Images larger than $D_{max}$ are downscaled proportionally.

In [13]:
def load_reference_image(image_path: Path, max_dimension: int = 800) -> Image.Image:
    """
    Load and normalize a reference image for use in the flash card.

    Parameters
    ----------
    image_path : Path
        Path to the reference image.
    max_dimension : int
        Maximum width or height in pixels after resizing.

    Returns
    -------
    PIL.Image.Image
        Resized image in RGB mode.
    """
    img = Image.open(image_path).convert("RGB")
    W, H = img.size

    scale = min(1.0, max_dimension / max(W, H))
    if scale < 1.0:
        new_size = (int(W * scale), int(H * scale))
        img = img.resize(new_size, Image.LANCZOS)

    return img

## 6. Unified ingestion function

The `ingest` function is the single entry point for the pipeline. It accepts
a file path, routes it through the correct branch, and returns a standardized
`IngestedDocument` regardless of input type.

The `IngestedDocument` dataclass unifies the four branch outputs:

| Field | Type | Description |
|---|---|---|
| `input_type` | `InputType` | classified input type |
| `text` | `str` | extracted text; empty for reference images |
| `reference_images` | `list` | PIL images; empty for text inputs |
| `source_path` | `Path` | original file path for traceability |

In [14]:
@dataclass
class IngestedDocument:
    """
    Standardized output of the ingestion pipeline.

    Attributes
    ----------
    input_type : InputType
        The classified input type.
    text : str
        Extracted text content. Empty string for reference images.
    reference_images : list[Image.Image]
        Reference images to embed in the flash card. Empty for non-image inputs.
    source_path : Path
        Original file path for traceability.
    """
    input_type       : InputType
    text             : str
    reference_images : list = field(default_factory=list)
    source_path      : Optional[Path] = None


def ingest(file_path: Path, is_reference_image: bool = False) -> IngestedDocument:
    """
    Unified ingestion entry point. Routes the file through the correct
    branch and returns a standardized IngestedDocument.

    Parameters
    ----------
    file_path : Path
        Path to the file to ingest.
    is_reference_image : bool
        If True and the file is an image, treat as a reference image
        rather than handwritten notes.

    Returns
    -------
    IngestedDocument
        Standardized ingestion output.
    """
    input_type = classify_input(file_path, is_reference_image)

    if input_type == InputType.PDF:
        text = extract_text_from_pdf(file_path)
        return IngestedDocument(input_type=input_type, text=text, source_path=file_path)

    elif input_type == InputType.HANDWRITTEN:
        text = extract_text_from_image(file_path)
        return IngestedDocument(input_type=input_type, text=text, source_path=file_path)

    elif input_type == InputType.REFERENCE_IMG:
        img = load_reference_image(file_path)
        return IngestedDocument(input_type=input_type, text="", reference_images=[img], source_path=file_path)

    elif input_type == InputType.PLAIN_TEXT:
        text = file_path.read_text(encoding="utf-8")
        return IngestedDocument(input_type=input_type, text=text, source_path=file_path)

## 7. Output validation

Before passing an `IngestedDocument` to the language model, we validate that
the extraction produced usable content. An empty or near-empty text string
indicates a failed extraction and should be caught here rather than producing
a malformed flash card downstream.

A document is valid if:
- Text-bearing inputs (`PDF`, `HANDWRITTEN`, `PLAIN_TEXT`) produce at least
  $n_{min}$ characters of text.
- Reference images produce at least one loaded `PIL.Image` object.

In [15]:
class IngestionError(Exception):
    pass


def validate_ingestion(doc: IngestedDocument, min_chars: int = 50) -> None:
    """
    Validate that an IngestedDocument contains usable content.

    Parameters
    ----------
    doc : IngestedDocument
        The document to validate.
    min_chars : int
        Minimum number of characters required in the text field
        for text-bearing input types.

    Raises
    ------
    IngestionError
        If the document fails validation.
    """
    text_bearing = {InputType.PDF, InputType.HANDWRITTEN, InputType.PLAIN_TEXT}

    if doc.input_type in text_bearing:
        if len(doc.text.strip()) < min_chars:
            raise IngestionError(
                f"Extraction produced insufficient text ({len(doc.text.strip())} chars) "
                f"from {doc.source_path}. "
                f"For PDFs: the file may be scanned. "
                f"For images: photo quality may be too low."
            )

    if doc.input_type == InputType.REFERENCE_IMG:
        if len(doc.reference_images) == 0:
            raise IngestionError(f"Reference image failed to load: {doc.source_path}")

## 8. Demo — Organic chemistry PDF and reference image

This demo processes two real inputs through the unified ingestion pipeline:
1. A chapter on carboxylic acid derivatives from OpenStax Organic Chemistry (PDF)
2. An image of an amide molecular structure (reference image)

### Getting the PDF
Download the full book from:
```
https://openstax.org/details/books/organic-chemistry
```
Then extract chapter 21 (pages 741 to 792) using PyMuPDF:

```python
import fitz
doc = fitz.open("openstax_organic_chemistry.pdf")
sub = fitz.open()
sub.insert_pdf(doc, from_page=740, to_page=791)
sub.save("data/uploads/openstax_ch21_carboxylic_acid_derivatives.pdf")
```

Note: PyMuPDF uses zero-based page indexing, so page 741 corresponds to index 740.

Both inputs are validated before being passed to downstream processing.

In [20]:
# --- Demo ---
from pathlib import Path

pdf_path = UPLOADS_DIR / "openstax_ch21_carboxylic_acid_derivatives.pdf"
img_path = UPLOADS_DIR / "amide_structure.jpg"

# Process PDF
doc_pdf = ingest(pdf_path)
validate_ingestion(doc_pdf)
print(f"PDF input type  : {doc_pdf.input_type.value}")
print(f"Characters      : {len(doc_pdf.text)}")
print(f"\n--- Text preview (first 500 chars) ---")
print(doc_pdf.text[:500])

# Process reference image
doc_img = ingest(img_path, is_reference_image=True)
validate_ingestion(doc_img)
print(f"\nImage input type : {doc_img.input_type.value}")
print(f"Images loaded    : {len(doc_img.reference_images)}")
print(f"Image size       : {doc_img.reference_images[0].size}")

PDF input type  : pdf
Characters      : 72868

--- Text preview (first 500 chars) ---
--- Page 1 ---
20.7 • Chemistry of Nitriles 723

Some general reactions of nitriles are shown in FIGURE 20.4.

FIGURE 20.4 Some reactions of nitriles.

Hydrolysis: Conversion of Nitriles into Carboxylic Acids Among the most useful reactions of nitriles is their hydrolysis to yield first an amide and then a carboxylic acid plus ammonia or an amine. The reaction occurs in either basic or acidic aqueous solution:

As shown in FIGURE 20.5, base-catalyzed nitrile hydrolysis involves nucleophilic addi

Image input type : reference_image
Images loaded    : 1
Image size       : (800, 211)
